In [4]:
from pathlib import Path
import zipfile
import pandas as pd


In [3]:
raw_folder = Path("../Data/raw")

t100_files = sorted(raw_folder.glob("t100_dec_*.zip"))

t100_files

[WindowsPath('../Data/raw/t100_dec_2022.zip'),
 WindowsPath('../Data/raw/t100_dec_2023.zip'),
 WindowsPath('../Data/raw/t100_dec_2024.zip'),
 WindowsPath('../Data/raw/t100_dec_2025.zip')]

In [5]:
for file in t100_files:
    with zipfile.ZipFile(file, "r") as z:
        print(file.name)
        print(z.namelist())
        print()

t100_dec_2022.zip
['T_T100D_SEGMENT_US_CARRIER_ONLY.csv']

t100_dec_2023.zip
['T_T100D_SEGMENT_US_CARRIER_ONLY.csv']

t100_dec_2024.zip
['T_T100D_SEGMENT_US_CARRIER_ONLY.csv']

t100_dec_2025.zip
['T_T100D_SEGMENT_US_CARRIER_ONLY.csv']



In [6]:
df_2022 = pd.read_csv(
    t100_files[0],
    compression = "zip"
)

df_2022.shape

(35734, 19)

In [7]:
df_2022.columns.tolist()

['DEPARTURES_SCHEDULED',
 'DEPARTURES_PERFORMED',
 'SEATS',
 'PASSENGERS',
 'DISTANCE',
 'UNIQUE_CARRIER',
 'UNIQUE_CARRIER_NAME',
 'ORIGIN_AIRPORT_ID',
 'ORIGIN',
 'ORIGIN_CITY_NAME',
 'ORIGIN_STATE_ABR',
 'ORIGIN_STATE_NM',
 'DEST_AIRPORT_ID',
 'DEST',
 'DEST_CITY_NAME',
 'DEST_STATE_ABR',
 'DEST_STATE_NM',
 'YEAR',
 'MONTH']

In [8]:
sea_2022 = df_2022[df_2022["ORIGIN"] == "SEA"].copy()

sea_2022.shape

(590, 19)

In [9]:
sea_2022["ORIGIN"].value_counts()

ORIGIN
SEA    590
Name: count, dtype: int64

In [10]:
t100_all = pd.concat(
    [pd.read_csv(file, compression = "zip") for file in t100_files],
    ignore_index = True
)

t100_all.shape

(149277, 19)

In [11]:
t100_all.groupby(["YEAR", "MONTH"]).size()

YEAR  MONTH
2022  12       35734
2023  12       36082
2024  12       38086
2025  12       39375
dtype: int64

In [12]:
sea_all = t100_all[t100_all["ORIGIN"] == "SEA"].copy()

sea_all.shape

(2431, 19)

In [13]:
sea_all["YEAR"].value_counts().sort_index()

YEAR
2022    590
2023    570
2024    649
2025    622
Name: count, dtype: int64

In [16]:
destination_demand = (
    sea_all
    .groupby(
        ["DEST", "DEST_CITY_NAME", "DEST_STATE_ABR"],
        as_index = False
    )["PASSENGERS"]
     .sum()
     .sort_values("PASSENGERS", ascending = False)
)

destination_demand.head(10)

,DEST,DEST_CITY_NAME,DEST_STATE_ABR,PASSENGERS
61,LAX,"Los Angeles, CA",CA,315904.0
89,PHX,"Phoenix, AZ",AZ,308110.0
60,LAS,"Las Vegas, NV",NV,299459.0
3,ANC,"Anchorage, AK",AK,278351.0
24,DEN,"Denver, CO",CO,269743.0
108,SFO,"San Francisco, CA",CA,261444.0
25,DFW,"Dallas/Fort Worth, TX",TX,237870.0
85,ORD,"Chicago, IL",IL,205165.0
101,SAN,"San Diego, CA",CA,194524.0
87,PDX,"Portland, OR",OR,178619.0


In [18]:
top10_airports = destination_demand.head(10)["DEST"].tolist()

top10_airports

['LAX', 'PHX', 'LAS', 'ANC', 'DEN', 'SFO', 'DFW', 'ORD', 'SAN', 'PDX']

In [21]:
top10_yearly = (
    sea_all[sea_all["DEST"].isin(top10_airports)]
    .groupby(["YEAR", "DEST"], as_index = False)["PASSENGERS"]
    .sum()
)

top10_yearly.head(10)

,YEAR,DEST,PASSENGERS
0,2022,ANC,70370.0
1,2022,DEN,65105.0
2,2022,DFW,45928.0
3,2022,LAS,75279.0
4,2022,LAX,73103.0
5,2022,ORD,51337.0
6,2022,PDX,50963.0
7,2022,PHX,72822.0
8,2022,SAN,47280.0
9,2022,SFO,64780.0


In [23]:
top10_pivot = top10_yearly.pivot(
    index = "DEST",
    columns = "YEAR",
    values = "PASSENGERS"
)

top10_pivot

YEAR,2022,2023,2024,2025
DEST,,,,
ANC,70370.0,69302.0,75953.0,62726.0
DEN,65105.0,69994.0,69185.0,65459.0
DFW,45928.0,52328.0,69731.0,69883.0
LAS,75279.0,80418.0,76896.0,66866.0
LAX,73103.0,77362.0,89868.0,75571.0
ORD,51337.0,46471.0,54986.0,52371.0
PDX,50963.0,46781.0,42970.0,37905.0
PHX,72822.0,77845.0,79482.0,77961.0
SAN,47280.0,48871.0,52532.0,45841.0


In [24]:
top10_pivot["CHANGE_2022_2025_PCT"] = (
    (top10_pivot[2025] - top10_pivot[2022])
    / top10_pivot[2022]
    * 100
).round(1)

top10_pivot.sort_values(
    "CHANGE_2022_2025_PCT",
    ascending=False
)

YEAR,2022,2023,2024,2025,CHANGE_2022_2025_PCT
DEST,,,,,
DFW,45928.0,52328.0,69731.0,69883.0,52.2
PHX,72822.0,77845.0,79482.0,77961.0,7.1
LAX,73103.0,77362.0,89868.0,75571.0,3.4
ORD,51337.0,46471.0,54986.0,52371.0,2.0
SFO,64780.0,64217.0,66690.0,65757.0,1.5
DEN,65105.0,69994.0,69185.0,65459.0,0.5
SAN,47280.0,48871.0,52532.0,45841.0,-3.0
ANC,70370.0,69302.0,75953.0,62726.0,-10.9
LAS,75279.0,80418.0,76896.0,66866.0,-11.2


In [25]:
top10_availability = (
    sea_all[sea_all["DEST"].isin(top10_airports)]
    .groupby(["DEST"], as_index=False)
    .agg(
        PASSENGERS=("PASSENGERS", "sum"),
        FLIGHTS=("DEPARTURES_PERFORMED", "sum"),
        SEATS=("SEATS", "sum")
    )
)

top10_availability

,DEST,PASSENGERS,FLIGHTS,SEATS
0,ANC,278351.0,2362.0,331528.0
1,DEN,269743.0,2001.0,312259.0
2,DFW,237870.0,1497.0,268959.0
3,LAS,299459.0,2051.0,350437.0
4,LAX,315904.0,2493.0,373393.0
5,ORD,205165.0,1442.0,239114.0
6,PDX,178619.0,2401.0,238323.0
7,PHX,308110.0,2375.0,372050.0
8,SAN,194524.0,1443.0,233094.0
9,SFO,261444.0,2181.0,321273.0


In [26]:
top10_availability["LOAD_FACTOR_PCT"] = (
    top10_availability["PASSENGERS"]
    / top10_availability["SEATS"]
    * 100
).round(1)

top10_availability.sort_values(
    "LOAD_FACTOR_PCT",
    ascending=False
)

,DEST,PASSENGERS,FLIGHTS,SEATS,LOAD_FACTOR_PCT
2,DFW,237870.0,1497.0,268959.0,88.4
1,DEN,269743.0,2001.0,312259.0,86.4
5,ORD,205165.0,1442.0,239114.0,85.8
3,LAS,299459.0,2051.0,350437.0,85.5
4,LAX,315904.0,2493.0,373393.0,84.6
0,ANC,278351.0,2362.0,331528.0,84.0
8,SAN,194524.0,1443.0,233094.0,83.5
7,PHX,308110.0,2375.0,372050.0,82.8
9,SFO,261444.0,2181.0,321273.0,81.4
6,PDX,178619.0,2401.0,238323.0,74.9


In [27]:
airline_service = (
    sea_all[sea_all["DEST"].isin(top10_airports)]
    .groupby(["UNIQUE_CARRIER_NAME"], as_index=False)
    .agg(
        PASSENGERS=("PASSENGERS", "sum"),
        FLIGHTS=("DEPARTURES_PERFORMED", "sum")
    )
    .sort_values("PASSENGERS", ascending=False)
)

airline_service

,UNIQUE_CARRIER_NAME,PASSENGERS,FLIGHTS
2,Alaska Airlines Inc.,1223112.0,8622.0
5,Delta Air Lines Inc.,503921.0,4133.0
16,United Air Lines Inc.,241239.0,1685.0
3,American Airlines Inc.,197142.0,1241.0
14,Southwest Airlines Co.,137970.0,986.0
13,SkyWest Airlines Inc.,107108.0,1798.0
8,Frontier Airlines Inc.,56696.0,323.0
9,Horizon Air,44208.0,716.0
15,Spirit Air Lines,34999.0,213.0
6,Envoy Air,2625.0,38.0


In [ ]:
passenger_airline_service = (
    sea_all[
        (sea_all["DEST"].isin(top10_airports)) &
        (sea_all["PASSENGERS"] > 0)
    ]
    .groupby(["UNIQUE_CARRIER_NAME"], as_index=False)
    .agg(
        PASSENGERS=("PASSENGERS", "sum"),
        FLIGHTS=("DEPARTURES_PERFORMED", "sum")
    )
    .sort_values("PASSENGERS", ascending=False)
)

passenger_airline_service